In [3]:
from openai import OpenAI

LLM_MODEL = "gpt-5-nano"
# LLM_MODEL = "gpt-4o-mini"

In [ ]:
client = OpenAI()
response = client.responses.create(
    model="gpt-4o-mini",
    input="Write a short bedtime story about a unicorn."
)

In [ ]:
client = OpenAI()
messages = [
    {"role": "user", "content": "Tell me a joke about Alexey."}
]

response = client.responses.create(
    model=LLM_MODEL,
    input=messages,
)

In [ ]:
response

In [ ]:
print(response.model_dump_json(indent=4))

In [ ]:
response.output_text

In [ ]:
response.output[0].content[0].text

In [ ]:
client = OpenAI()

messages = [
    {"role": "user", "content": "Tell me story about unicorn."}
]

response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
)

In [ ]:
print(response.output_text)

In [ ]:
print(response.usage.model_dump_json(indent=4))

In [1]:
# prices for 1 million tokens in dollars
MODEL_PRICES = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-5-nano": {"input": 0.20, "output": 1.25},
    # "gpt-5-mini": {"input": 0.25, "output": 2.00},
    # "gpt-5.2": {"input": 1.75, "output": 14.00},
    # "gpt-5.2-pro": {"input": 21.00, "output": 168.00},
}

def calculate_cost(llm_model: str, input_tokens: int, output_tokens: int) -> None:
    prices = MODEL_PRICES[llm_model]
    input_cost = 1e-6 * input_tokens * prices["input"]
    output_cost = 1e-6 * output_tokens * prices["output"]
    total_cost = input_cost + output_cost

    # print(f"Input cost: ${input_cost:.6f}")
    # print(f"Output cost: ${output_cost:.6f}")
    # print(f"Total cost: ${total_cost:.6f}")

    label_w = 12
    value_w = 10  # includes digits + dot

    print("=== cost calculation ===")
    print(f"{'Input cost:':<{label_w}} {f'${input_cost:.6f}':>{value_w}}")
    print(f"{'Output cost:':<{label_w}} {f'${output_cost:.6f}':>{value_w}}")
    print(f"{'Total cost:':<{label_w}} {f'${total_cost:.6f}':>{value_w}}")
    print("=" * (label_w + value_w + 2))

In [4]:
client = OpenAI()

messages = [
    {"role": "user", "content": "Tell me story about unicorn."}
]

response = client.responses.create(
    model=LLM_MODEL,
    input=messages,
)

calculate_cost(
    llm_model=LLM_MODEL,
    input_tokens=response.usage.input_tokens,
    output_tokens=response.usage.output_tokens,
)


=== cost calculation ===
Input cost:   $0.000002
Output cost:  $0.007311
Total cost:   $0.007314


In [ ]:
client = OpenAI()

messages = [
    {"role": "user", "content": "Tell me story about unicorn."}
]

stream = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    stream=True,
)

for event in stream:
    if hasattr(event, "delta"):
        print(event.delta, end="")

In [ ]:
client = OpenAI()

messages = [
    {"role": "user", "content": "Tell me story about unicorn."}
]

stream = client.responses.create(
    model=LLM_MODEL,
    input=messages,
    stream=True,
)

events = []
for event in stream:
    events.append(event)
    if hasattr(event, "delta"):
        print(event.delta, end="")

In [6]:
len(events)

1824

In [ ]:
# events[-1].response.output[1].content[0].text
events[-1].response.output_text

In [ ]:
for event in events:
    # if event.type == "response.output_text.delta":
    if hasattr(event, "delta"):
        continue
        print(event.delta, end="")
    if event.type == "response.completed":
    # if hasattr(event, "response") and event.response.usage is not None:
        # print("-" * 20, event.type)
        # print(event.model_dump_json(indent=4))
        print(end="\n\n")
        calculate_cost(
            llm_model=LLM_MODEL,
            input_tokens=event.response.usage.input_tokens,
            output_tokens=event.response.usage.output_tokens,
        )


## System prompt

In [ ]:
client = OpenAI()

SYSTEM_PROMPT = "Make sure to include an emojies in the story."
USER_PROMPT = "Tell me story about unicorn."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT}
]

response = client.responses.create(
    model=LLM_MODEL,
    input=messages,
)

print(response.output_text)

calculate_cost(
    llm_model=LLM_MODEL,
    input_tokens=response.usage.input_tokens,
    output_tokens=response.usage.output_tokens,
)

## LLMs are Stateless

In [44]:
client = OpenAI()

SYSTEM_PROMPT = "Make sure to include an emojies in replies."
USER_PROMPT_1 = "My name is rmblx."
USER_PROMPT_2 = "What is my name?"

messages_1 = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT_1},
]

response = client.responses.create(
    model=LLM_MODEL,
    input=messages_1,
)

print(response.output_text)

messages_2 = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT_2},
]

response = client.responses.create(
    model=LLM_MODEL,
    input=messages_2,
)
print(response.output_text)


Nice to meet you, rmblx! 👋 What would you like to do today? I can help with brainstorming, writing, coding, learning new topics, or anything else you have in mind. 😊✨
I don’t know your name yet. If you tell me, I’ll use it in this chat. What would you like me to call you? You can share your real name or a nickname, and I can remember it for this conversation. 😊👋


## Memory

In [54]:
client = OpenAI()

def add_system_message(messages: list[dict], content: str) -> None:
    messages.append({"role": "system", "content": content})

def add_user_message(messages: list[dict], content: str) -> None:
    messages.append({"role": "user", "content": content})

def add_assistant_message(messages: list[dict], content: str) -> None:
    messages.append({"role": "assistant", "content": content})

def chat(messages: list[dict], user_input: str, client: OpenAI) -> str:
    add_user_message(messages, user_input)
    response = client.responses.create(
        model=LLM_MODEL,
        input=messages,
    )
    add_assistant_message(messages, response.output_text)
    return response.output_text


add_assistant_message(messages, "Answer in one sentence.")

llm_response = chat(messages, "My name is rmblx.", client)
print(llm_response)

llm_response = chat(messages, "What is my name?", client)
print(llm_response)

llm_response = chat(messages, "What is a capital of France?", client)
print(llm_response)

llm_response = chat(messages, "What was the country I was asking about?", client)
print(llm_response)


Nice to meet you, rmblx 🦄.
Your name is rmblx 🦄.
The capital of France is Paris 🗼.
The country you were asking about was France 🇫🇷.
